# UVERA · NB07 — AI-7 Grounded Brief (Gemini)

| | |
|---|---|
| **Who runs it** | Third runner (see guide) |
| **Accelerator** | None (CPU) |
| **Internet** | On |
| **Secret** | `GEMINI_API_KEY` (Add-ons → Secrets) |
| **Inputs to add** | — (uses AI-1 and AI-6 results from the repo; run **after** NB01 and NB06 results are committed) |
| **Expected runtime** | ~20–40 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** Gemini turns the model's evidence into two to four plain sentences in Bangla and English. It is never allowed to decide anything. A strict checker rejects any text that adds numbers or reasons not in the evidence, promises safety, contradicts a high risk, or hides uncertainty; rejected text is replaced by a fixed template. We also attack it with prompt-injection messages to prove unsafe text never reaches a user.

**Outputs** (in `/kaggle/working/outputs/NB07_outputs.zip`): `artifacts/ai7/briefs_cache.json`, `reports/metrics_ai7.json`, `reports/ai7_results.csv`

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)

In [ ]:
!pip install -q google-genai

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=90)
OUT = out_dir("NB07")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb07.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# 3) Build the synthetic world. Same seed -> identical data on every Kaggle account (checked by hash).
from pathlib import Path
from uvera_ml.sim.world import load_or_generate
world = load_or_generate("/tmp/world_full", scale="full")
print(json.dumps(world.meta, indent=1)[:1500])
ref = Path("/tmp/uvera/reports/world_meta.json")
if ref.exists():
    expected = json.loads(ref.read_text(encoding="utf-8"))["meta"]["hashes"]["events"]
    print("Data identical to NB00 run:", expected == world.meta["hashes"]["events"])
budget.check("world")

In [ ]:
import pandas as pd
from pathlib import Path
from uvera_ml.common import read_json
base = Path("/tmp/uvera/artifacts")
if (base / "ai1" / "test_scores.parquet").exists():
    scored = pd.read_parquet(base / "ai1" / "test_scores.parquet"); thr = read_json(base / "ai1" / "thresholds.json")
else:
    from uvera_ml.models.ai1 import run_ai1
    run_ai1(world, "/tmp/ai1_quick", quick=True)
    scored = pd.read_parquet("/tmp/ai1_quick/artifacts/ai1/test_scores.parquet"); thr = read_json("/tmp/ai1_quick/artifacts/ai1/thresholds.json")
cases = json.loads((base / "ai6" / "cases.json").read_text(encoding="utf-8")) if (base / "ai6" / "cases.json").exists() else []
print(len(scored), "scored transfers,", len(cases), "cases")

In [ ]:
from kaggle_secrets import UserSecretsClient
from uvera_ml.brief.gemini import Gemini
from uvera_ml.models.ai7 import run_ai7
try:
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
    gem = Gemini(cache_dir="/kaggle/working/gemini_cache")
except Exception as e:
    gem = None; print("Gemini not available, template-only run:", e)
summary = run_ai7(OUT, scored, thr, cases, gemini=gem, n_eval=200, n_injection=30)
print(json.dumps(summary, indent=1, default=float))
cache = json.loads((OUT / "artifacts" / "ai7" / "briefs_cache.json").read_text(encoding="utf-8"))
for v in list(cache.values())[:3]:
    print(v["brief"]["english"]); print(v["brief"]["bangla"]); print("---")
budget.check("ai7")

In [ ]:
# Last step: package the results. Download the zip from the Output tab and send it to Rubab.
budget.check("done")
zip_outputs(OUT)